In [1]:
import pandas as pd
import openai
import seaborn as sns

In [ ]:
API_KEY = "sk-H-................"
BASE_URL = "https://api.futurexailab.com"
client = openai.OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [3]:
df = sns.load_dataset("tips")
df = df[['sex', 'smoker', 'day', 'time', 'tip', 'total_bill']].dropna()
df.head()

,sex,smoker,day,time,tip,total_bill
0,Female,No,Sun,Dinner,1.01,16.99
1,Male,No,Sun,Dinner,1.66,10.34
2,Male,No,Sun,Dinner,3.50,21.01
3,Male,No,Sun,Dinner,3.31,23.68
4,Female,No,Sun,Dinner,3.61,24.59


In [4]:
summary = df.groupby(['sex', 'smoker']).agg(
    count=('tip', 'size'),
    avg_tip=('tip', 'mean'),
    avg_bill=('total_bill', 'mean'),
    tip_rate=('tip', lambda x: (x.mean() / df['tip'].mean()))  # relative comparison
).reset_index()

summary

/tmp/ipython-input-1211979369.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = df.groupby(['sex', 'smoker']).agg(


,sex,smoker,count,avg_tip,avg_bill,tip_rate
0,Male,Yes,60,3.051167,22.284500,1.017639
1,Male,No,97,3.113402,19.791237,1.038396
2,Female,Yes,33,2.931515,17.977879,0.977733
3,Female,No,54,2.773519,18.105185,0.925037


In [5]:
prompt_text = f'''
I have summarized a restaurant tipping dataset. Columns:
- sex: gender of customer
- smoker: whether the customer is a smoker
- count: number of customers in this group
- avg_tip: average tip amount for this group
- avg_bill: average bill amount
- tip_rate: group's tip rate relative to overall average

Data summary:
{summary.to_dict(orient='records')}

Analyze this summary and:
1. Detect potential bias in tipping behavior based on gender and smoker status.
2. Highlight any fairness concerns in tipping trends.
3. Suggest how to reduce or interpret these biases properly.
'''


In [6]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": "You are a data fairness and AI bias expert."},
        {"role": "user", "content": prompt_text}
    ],
    temperature=0
)

bias_analysis = response.choices[0].message.content
bias_analysis


'This is an insightful summary of tipping behavior, and it allows us to identify some interesting trends and potential biases.\n\nLet\'s break down the analysis:\n\n### 1. Detection of Potential Bias in Tipping Behavior\n\nThe `tip_rate` column is crucial here, as it normalizes the tipping behavior relative to the overall average, making comparisons across groups more meaningful.\n\n*   **Overall Gender Trend:**\n    *   **Males (both smoker and non-smoker groups)** consistently show a `tip_rate` **above 1.0** (1.017 and 1.038), indicating they tip *more* than the overall average.\n    *   **Females (both smoker and non-smoker groups)** consistently show a `tip_rate` **below 1.0** (0.977 and 0.925), indicating they tip *less* than the overall average.\n    *   **Conclusion:** There is a clear trend suggesting **males tend to tip more generously than females** in this dataset, relative to the overall average.\n\n*   **Smoker Status Trend (within gender):**\n    *   **Among Males:** Non-